In [1]:
import pandas as pd

print("⏳ Uniendo las piezas del rompecabezas desde '../data/01_raw/'...")

try:
    # 1. Leer los archivos usando tu ruta exacta con '../'
    df_demo = pd.read_csv("../data/01_raw/demographics.csv")
    df_meas = pd.read_csv("../data/01_raw/body_measures_real.csv")
    df_lab = pd.read_json("../data/01_raw/laboratory.json")

    # 2. Unirlos todos en el Dataset Maestro en bruto
    df_master = pd.merge(df_demo, df_meas, on="SEQN", how="inner")
    df_master = pd.merge(df_master, df_lab, on="SEQN", how="inner")

    print("\n ==================================================")
    print("       DIAGNÓSTICO DEL DATASET MAESTRO (df_master)    ")
    print("====================================================")
    print(f"🔹 Filas totales en bruto: {df_master.shape[0]}")
    print(f"🔹 Columnas totales: {df_master.shape[1]}")
    print("\n🔄 Pacientes por ciclo confirmados:")
    print(df_master['SDDSRVYR'].value_counts().sort_index())

    # ==============================================================================
    # FASE 1: FILTRADO DE LA POBLACIÓN OBJETIVO (ADULTOS >= 18 AÑOS)
    # ==============================================================================
    print("\n🔞 ==================================================")
    print("     FASE 1: FILTRADO DE POBLACIÓN OBJETIVO (ADULTOS) ")
    print("====================================================")
    
    df_adultos = df_master[df_master['RIDAGEYR'] >= 18].copy()
    
    print(f"✅ ¡Filtro aplicado con éxito!")
    print(f" Tu nueva población de adultos para el modelo es de: {df_adultos.shape[0]} pacientes.")

except Exception as e:
    print(f"\n❌ Error al unir los datos: {e}")

⏳ Uniendo las piezas del rompecabezas desde '../data/01_raw/'...

       DIAGNÓSTICO DEL DATASET MAESTRO (df_master)    
🔹 Filas totales en bruto: 31568
🔹 Columnas totales: 155

🔄 Pacientes por ciclo confirmados:
SDDSRVYR
7.0     7821
8.0     8291
9.0     8021
10.0    7435
Name: count, dtype: int64

🔞 ==================================================
     FASE 1: FILTRADO DE POBLACIÓN OBJETIVO (ADULTOS) 
✅ ¡Filtro aplicado con éxito!
 Tu nueva población de adultos para el modelo es de: 22807 pacientes.


In [2]:
# ==============================================================================
# FASE 2: SELECCIÓN DE VARIABLES CRÍTICAS (FEATURE SELECTION)
# ==============================================================================
variables_clave = [
    "SEQN", "RIAGENDR", "RIDAGEYR", "RIDRETH3", "DMDEDUC2", "DMDMARTL", 
    "INDFMPIR", "BMXWT", "BMXHT", "BMXBMI", "BMXWAIST", 
    "BPXSY1", "BPXSY2", "BPXSY3", "BPXDI1", "BPXDI2", "BPXDI3", 
    "LBXTC", "SMQ020"
]
df_salud = df_adultos[variables_clave].copy()


# ==============================================================================
# FASE 3: TRATAMIENTO DE VALORES NULOS E IMPUTACIÓN
# ==============================================================================

# 3.1 Filtro Físico Base (Sacar registros sin Peso o Altura)
df_salud = df_salud.dropna(subset=['BMXWT', 'BMXHT']).copy()

# 3.2 Filtro Estricto de Presión Arterial (Mínimo 2 tomas válidas)
df_salud['TOMAS_SISTOLICA'] = df_salud[['BPXSY1', 'BPXSY2', 'BPXSY3']].notnull().sum(axis=1)
df_salud['TOMAS_DIASTOLICA'] = df_salud[['BPXDI1', 'BPXDI2', 'BPXDI3']].notnull().sum(axis=1)

# Filtrar: Solo nos quedamos con quienes tengan al menos 2 mediciones de cada una
df_salud = df_salud[(df_salud['TOMAS_SISTOLICA'] >= 2) & (df_salud['TOMAS_DIASTOLICA'] >= 2)].copy()

# Calcular el promedio de las presiones válidas
df_salud['BPXSY_MEAN'] = df_salud[['BPXSY1', 'BPXSY2', 'BPXSY3']].mean(axis=1).round(2)
df_salud['BPXDI_MEAN'] = df_salud[['BPXDI1', 'BPXDI2', 'BPXDI3']].mean(axis=1).round(2)

# Limpiar las columnas individuales de presión que ya no usaremos
df_salud = df_salud.drop(columns=[
    'BPXSY1', 'BPXSY2', 'BPXSY3', 'BPXDI1', 'BPXDI2', 'BPXDI3', 
    'TOMAS_SISTOLICA', 'TOMAS_DIASTOLICA'
])

# 3.3 Imputación de Variables Continuas (Rellenar con la Mediana)
# Se usa para el Ratio de Ingresos, Circunferencia de Cintura y Colesterol Total
for col in ['INDFMPIR', 'BMXWAIST', 'LBXTC']:
    df_salud[col] = df_salud[col].fillna(df_salud[col].median())

# 3.4 Imputación de Variables Categóricas (Rellenar con la Moda)
# Se usa para el Nivel Educativo y el Estado Civil
for col in ['DMDEDUC2', 'DMDMARTL']:
    df_salud[col] = df_salud[col].fillna(df_salud[col].mode()[0])

# 3.5 Eliminación Preventiva de Duplicados
df_salud = df_salud.drop_duplicates(subset=['SEQN'], keep='first')

print("🧼 ==================================================")
print(f"✅ ¡Limpieza de nulos completada con éxito!")
print(f"📊 Número de pacientes finales: {df_salud.shape[0]}")
print("====================================================")

🧼 ==================================================
✅ ¡Limpieza de nulos completada con éxito!
📊 Número de pacientes finales: 21463


In [3]:
# ==============================================================================
# FASE 4: TRADUCCIÓN DE VARIABLES (MAPEO DE CATEGORÍAS)
# ==============================================================================

# Diccionarios de traducción basados en los libros de códigos de la NHANES
dict_genero = {1.0: 'Masculino', 2.0: 'Femenino'}
dict_etnia = {1.0: 'Mexicano Americano', 2.0: 'Otro Hispano', 3.0: 'Blanco', 4.0: 'Negro', 6.0: 'Asiático', 7.0: 'Otra/Multirracial'}
dict_educacion = {1.0: 'Básica Incompleta', 2.0: 'Básica Completa', 3.0: 'Secundaria', 4.0: 'Técnico/Universitario Incompleto', 5.0: 'Universitario Completo'}
dict_estado_civil = {1.0: 'Casado', 2.0: 'Viudo', 3.0: 'Divorciado', 4.0: 'Separado', 5.0: 'Soltero', 6.0: 'Conviviente'}
dict_fumador = {1.0: 'Sí', 2.0: 'No'}

# Aplicar los reemplazos de códigos por texto amigable
df_salud['RIAGENDR'] = df_salud['RIAGENDR'].replace(dict_genero)
df_salud['RIDRETH3'] = df_salud['RIDRETH3'].replace(dict_etnia)
df_salud['DMDEDUC2'] = df_salud['DMDEDUC2'].replace(dict_educacion)
df_salud['DMDMARTL'] = df_salud['DMDMARTL'].replace(dict_estado_civil)
df_salud['SMQ020'] = df_salud['SMQ020'].replace(dict_fumador)

# Renombrar los encabezados de las columnas a español técnico
nuevos_nombres = {
    'SEQN': 'ID_Paciente', 'RIAGENDR': 'Genero', 'RIDAGEYR': 'Edad', 'RIDRETH3': 'Etnia',
    'DMDEDUC2': 'Educacion', 'DMDMARTL': 'Estado_Civil', 'INDFMPIR': 'Ratio_Ingresos',
    'BMXWT': 'Peso_kg', 'BMXHT': 'Estatura_cm', 'BMXBMI': 'IMC', 'BMXWAIST': 'Cintura_cm',
    'LBXTC': 'Colesterol_Total', 'SMQ020': 'Fumador_Historico',
    'BPXSY_MEAN': 'Presion_Sistolica', 'BPXDI_MEAN': 'Presion_Diastolica'
}
df_salud = df_salud.rename(columns=nuevos_nombres)

# Guardar el resultado en la carpeta intermedia usando la ruta relativa correcta
df_salud.to_csv("../data/02_intermediate/df_nhanes_adultos_limpio.csv", index=False)

print("💾 ==================================================")
print("✅ ¡Archivo maestro de analítica exportado correctamente!")
print("📍 Guardado en: '../data/02_intermediate/df_nhanes_adultos_limpio.csv'")
print("====================================================")
print(df_salud.head(3))

💾 ==================================================
✅ ¡Archivo maestro de analítica exportado correctamente!
📍 Guardado en: '../data/02_intermediate/df_nhanes_adultos_limpio.csv'
   ID_Paciente     Genero  Edad     Etnia                         Educacion  \
0        62161  Masculino  22.0    Blanco                        Secundaria   
2        62164   Femenino  44.0    Blanco  Técnico/Universitario Incompleto   
6        62169  Masculino  21.0  Asiático                        Secundaria   

  Estado_Civil  Ratio_Ingresos  Peso_kg  Estatura_cm   IMC  Cintura_cm  \
0      Soltero            3.15     69.2        172.3  23.3        81.0   
2       Casado            1.67     67.2        170.1  23.2        80.1   
6      Soltero            0.33     54.6        164.8  20.1        69.6   

   Colesterol_Total Fumador_Historico  Presion_Sistolica  Presion_Diastolica  
0             168.0                No             110.67               74.67  
2             190.0                No           

In [4]:
print("🔍 ==================================================")
print("      AUDITORÍA DE CALIDAD DEL DATASET FINAL         ")
print("====================================================")

# 1. Revisar valores nulos por columna
print("\n🛑 Cantidad de Valores Nulos por Columna:")
nulos = df_salud.isnull().sum()

# Si hay nulos, los muestra. Si no, lanza un mensaje de éxito.
if nulos.sum() > 0:
    print(nulos[nulos > 0])
else:
    print("✅ ¡Impecable! Cero nulos. Todas las celdas tienen datos.")

# 2. Revisar filas duplicadas (pacientes repetidos exactamente iguales)
print("\n👯‍♂️ Revisión de Filas Duplicadas:")
duplicados = df_salud.duplicated().sum()

if duplicados == 0:
    print("✅ ¡Perfecto! Cero duplicados. Cada paciente es único.")
else:
    print(f"⚠️ Atención: Se encontraron {duplicados} filas duplicadas.")

# 3. Resumen final de la estructura
print("\n📊 ==================================================")
print(f"Estructura final aprobada: {df_salud.shape[0]} pacientes y {df_salud.shape[1]} variables.")
print("====================================================")

🔍 ==================================================
      AUDITORÍA DE CALIDAD DEL DATASET FINAL         

🛑 Cantidad de Valores Nulos por Columna:
Fumador_Historico    275
dtype: int64

👯‍♂️ Revisión de Filas Duplicadas:
✅ ¡Perfecto! Cero duplicados. Cada paciente es único.

📊 ==================================================
Estructura final aprobada: 21463 pacientes y 15 variables.


In [5]:
# 1. Eliminar directamente las filas donde no sabemos si fuma o no
df_salud = df_salud.dropna(subset=['Fumador_Historico']).copy()

# 2. Guardar el archivo actualizado
df_salud.to_csv("../data/02_intermediate/df_nhanes_adultos_limpio.csv", index=False)

print(f" Pacientes con nulos eliminados.")
print(f"📊 Pacientes finales 100% reales: {df_salud.shape[0]}")

 Pacientes con nulos eliminados.
📊 Pacientes finales 100% reales: 21188


ID_Paciente           0
Genero                0
Edad                  0
Etnia                 0
Educacion             0
Estado_Civil          0
Ratio_Ingresos        0
Peso_kg               0
Estatura_cm           0
IMC                   0
Cintura_cm            0
Colesterol_Total      0
Fumador_Historico     0
Presion_Sistolica     0
Presion_Diastolica    0
dtype: int64
